# Loader walkthrough

This notebook shows how the new ingestion loaders behave on a few sample files.

It will:

- import the loader entrypoint
- pick one sample file from each supported format
- print the canonical block shape returned by the loader
- show a few metadata fields so you can sanity-check the output

In [1]:
import logging
logging.basicConfig(level=logging.DEBUG)

In [2]:
from pathlib import Path
from pprint import pprint

from ragforge.loader.loader import load_file

sample_root = Path('../../data/ingestion/sample')
sample_root.exists()

True

In [3]:
def first_file(folder: Path, suffixes: tuple[str, ...]) -> Path | None:
    if not folder.exists():
        return None
    for path in sorted(folder.iterdir()):
        if path.is_file() and path.suffix.lower() in suffixes:
            return path
    return None


sample_files = {
    'pdf': first_file(sample_root / 'pdf', ('.pdf',)),
    'pptx': first_file(sample_root / 'pptx', ('.pptx', '.ppt')),
    'txt': first_file(sample_root / 'txt', ('.txt', '.log')),
}

sample_files

{'pdf': PosixPath('../../data/ingestion/sample/pdf/2204.02809_hammer_pdf.pdf'),
 'pptx': PosixPath('../../data/ingestion/sample/pptx/cip_source.pptx'),
 'txt': PosixPath('../../data/ingestion/sample/txt/pg11_alice_in_wonderland.txt')}

In [4]:
# for kind, path in sample_files.items():
#     if path is None:
#         print(f'{kind}: no sample file found')
#         continue

#     print(f'\n=== {kind.upper()} ===')
#     print(f'file: {path}')
#     blocks = load_file(str(path))
#     print(f'blocks: {len(blocks)}')
#     if not blocks:
#         continue

#     first = blocks[0]
#     print('keys:', sorted(first.keys()))
#     print('metadata:')
#     pprint(first.get('metadata', {}))
#     print('text preview:')
#     print(first.get('text', '')[:500])

In [5]:
# Optional: inspect a single file of your choice.
# Replace the path below with any local PDF, PPTX, TXT, LOG, Excel, or image file.

# file_path = sample_files['pdf'] or sample_files['pptx'] or sample_files['txt']
# if file_path:
#     blocks = load_file(str(file_path))
#     pprint(blocks[0])
# else:
#     print('No sample file available yet.')

In [4]:
file_path = '/Users/sunnyraj/code_files/git_repos/RagForge/data/ingestion/sample/pdf/2501.03936_pptagent.pdf'
file_path = '/Users/sunnyraj/code_files/git_repos/RagForge/data/ingestion/sample/pdf/2311.01767_pptc_benchmark.pdf'
blocks = load_file(str(file_path))

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [5]:
blocks[1]

{'doc_id': 'da91f9abb5d215f1231bc400d453f39feb4141a492fef2226accd8870ffc4d71',
 'page_num': 2,
 'loading_strategy': 'text',
 'metadata': {'source': '/Users/sunnyraj/code_files/git_repos/RagForge/data/ingestion/sample/pdf/2311.01767_pptc_benchmark.pdf',
  'filename': '2311.01767_pptc_benchmark.pdf',
  'file_type': 'pdf',
  'page': 2,
  'ocr_run': False,
  'table_count': 0,
  'image_count': 1,
  'loading_strategy': 'text',
  'unit_kind': 'page',
  'unit_num': 2},
 'content': {'tables': [],
  'text': 'Figure 2: We illustrate how LLMs complete one turn in a session. (A) To prompt the LLM, we provide it with the current instruction, previous instructions (dialogue history), PPT file content, and the API reference file. ’PPT reader’ is a function that transforms the PPT file into the text-based format as the PPT file content. (B) The LLM then generates the API sequence and executes it to obtain the prediction PPT file. (C) We evaluate attributes and position relations in the prediction file.

In [8]:

import httpx

OLLAMA_URL = "http://localhost:11434"
model = "gemma4:e4b"
prompt = "hi"
response = httpx.post(
                f"{OLLAMA_URL}/api/generate",
                json={"model": model, "prompt": prompt, "stream": False},
                timeout=60.0,
            )
response.raise_for_status()
payload = response.json()

DEBUG:httpcore.connection:connect_tcp.started host='localhost' port=11434 local_address=None timeout=60.0 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x131fd4590>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'application/json; charset=utf-8'), (b'Date', b'Mon, 22 Jun 2026 13:15:44 GMT'), (b'Content-Length', b'434')])
INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"
DEBUG:httpcore.http11:receive_response_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:rece

In [9]:
payload

{'model': 'gemma4:e4b',
 'created_at': '2026-06-22T13:15:44.712531Z',
 'response': 'Hi! How can I help you today? 😊',
 'done': True,
 'done_reason': 'stop',
 'context': [2,
  105,
  9731,
  107,
  98,
  107,
  106,
  107,
  105,
  2364,
  107,
  2202,
  106,
  107,
  105,
  4368,
  107,
  10979,
  236888,
  2088,
  740,
  564,
  1601,
  611,
  3124,
  236881,
  103453],
 'total_duration': 5369221167,
 'load_duration': 265131292,
 'prompt_eval_count': 17,
 'prompt_eval_duration': 4685042000,
 'eval_count': 11,
 'eval_duration': 389865000}

In [6]:
from ragforge.ingestion.chunking.context_chunker import chunk_documents

In [7]:
chunked_docs = chunk_documents(blocks)

{"message": "Starting document chunking process, Total documents: 17", "logger": "ragforge.ingestion.chunking", "level": "INFO"}
{"event": "chunk_created", "doc_id": "da91f9abb5d215f1231bc400d453f39feb4141a492fef2226accd8870ffc4d71", "chunk_index": 0, "file_type": "pdf", "logger": "ragforge.ingestion.chunking", "level": "INFO"}
{"event": "chunk_created", "doc_id": "da91f9abb5d215f1231bc400d453f39feb4141a492fef2226accd8870ffc4d71", "chunk_index": 1, "file_type": "pdf", "logger": "ragforge.ingestion.chunking", "level": "INFO"}
{"event": "chunk_created", "doc_id": "da91f9abb5d215f1231bc400d453f39feb4141a492fef2226accd8870ffc4d71", "chunk_index": 2, "file_type": "pdf", "logger": "ragforge.ingestion.chunking", "level": "INFO"}
{"event": "chunk_created", "doc_id": "da91f9abb5d215f1231bc400d453f39feb4141a492fef2226accd8870ffc4d71", "chunk_index": 3, "file_type": "pdf", "logger": "ragforge.ingestion.chunking", "level": "INFO"}
{"event": "chunk_created", "doc_id": "da91f9abb5d215f1231bc400d453f

In [8]:
chunked_docs[0]

{'id': 'af5069ee-79d8-4989-a30a-865aff0c532f',
 'text': 'PPTC Benchmark: Evaluating Large Language Models for PowerPoint Task Completion Yiduo Guo1∗, Zekai Zhang1∗, Yaobo Liang2, Dongyan Zhao1, Nan Duan2 1Peking University 2Microsoft Research Asia yiduo@stu.|||multi-modal,zhang1,liang2,1peking,2microsoft,PPTC,language,models,powerpoint,completion,stu.pku.edu.cn,microsoft.com|||PPTC Benchmark: Evaluating Large Language Models for PowerPoint Task Completion Yiduo Guo1∗, Zekai Zhang1∗, Yaobo Liang2, Dongyan Zhao1, Nan Duan2 1Peking University 2Microsoft Research Asia yiduo@stu.pku.edu.cn,justinzzk@stu.pku.edu.cn,yaobo.liang@microsoft.com zhaody@pku.edu.cn,nanduan@microsoft.com Abstract Recent evaluations of Large Language Models (LLMs) have centered around testing their zero- shot/few-shot capabilities for basic natural lan- guage tasks and their ability to translate instruc- tions into tool APIs. However, the evaluation of LLMs utilizing complex tools to finish multi- turn, multi-modal i

In [9]:
from ragforge.ingestion.indexing import upsert_documents

/Users/sunnyraj/code_files/git_repos/RagForge/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
upsert_documents(collection_name="ragforge-test", documents=chunked_docs)

{"event": "upsert_documents_started", "collection": "ragforge-test", "num_docs": 95, "logger": "ragforge.ingestion.qdrant", "level": "INFO"}
DEBUG:httpcore.connection:connect_tcp.started host='localhost' port=6333 local_address=None timeout=5.0 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x13e7935f0>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'GET']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'GET']>
DEBUG:httpcore.connection:connect_tcp.started host='localhost' port=6333 local_address=None timeout=5 socket_options=None
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'GET']>
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x13e983b00>
DEBUG:httpcore.http11:send_request_head

"Successfully upserted 95 documents into 'ragforge-test'."

DEBUG:urllib3.connectionpool:Resetting dropped connection: localhost
DEBUG:urllib3.connectionpool:http://localhost:6006 "POST /v1/traces HTTP/1.1" 200 0


In [ ]:
import httpx
ollama_url = "http://localhost:11434"
model_id = "embeddinggemma:latest"
text = "This is a sample text to be embedded."
response = httpx.post(  
            f"{ollama_url}/api/embeddings",
            json={"model": model_id, "prompt": text},
            timeout=30.0,
        )
response.raise_for_status()
response.json()["embedding"]

DEBUG:httpcore.connection:connect_tcp.started host='localhost' port=11434 local_address=None timeout=30.0 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x14388fd70>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'application/json; charset=utf-8'), (b'Date', b'Mon, 22 Jun 2026 13:01:31 GMT'), (b'Transfer-Encoding', b'chunked')])
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embeddings "HTTP/1.1 200 OK"
DEBUG:httpcore.http11:receive_response_body.started request=<Request [b'POST']>
DEBUG:httpcore.ht

[-0.15397918224334717,
 -0.003218425437808037,
 -0.007352741435170174,
 -0.03284312039613724,
 -0.028697669506072998,
 0.013091818429529667,
 -0.021494179964065552,
 0.0285491906106472,
 0.022219402715563774,
 -0.010608099400997162,
 -0.02915976569056511,
 -0.059801314026117325,
 -0.004153704736381769,
 -0.01871769316494465,
 0.06745988130569458,
 -0.0413212813436985,
 0.04327714070677757,
 0.023307427763938904,
 -0.08542650192975998,
 0.008814260363578796,
 -0.0005485088331624866,
 0.033947963267564774,
 -0.03783029690384865,
 -0.029858706519007683,
 -0.02220163121819496,
 -0.009344184771180153,
 0.01707703433930874,
 -0.029112979769706726,
 -0.03345466032624245,
 0.014206316322088242,
 -0.04377416893839836,
 -0.03926600143313408,
 0.08578100055456161,
 0.021726449951529503,
 0.04378413036465645,
 0.03869134932756424,
 0.015825267881155014,
 -0.0659148171544075,
 0.033133141696453094,
 0.003450374584645033,
 -0.018609927967190742,
 0.06493722647428513,
 -0.03527064248919487,
 0.055997